# 5G NR Demodulation: Custom Code vs Keysight VSA 89600

This notebook generates a 5G NR PDSCH waveform, demodulates it with custom Python code,
then uses Keysight VSA 89600 (or simulated results in offline mode) and compares the two approaches.

**Pipeline:**
1. Generate 5G NR waveform (OFDM, DMRS, QAM)
2. Custom Python demodulation (OFDM demod → channel est → ZF equalization → hard decision)
3. VSA 89600 demodulation (COM automation or offline simulation)
4. Side-by-side comparison (EVM, BER, constellation, per-symbol metrics)

## 1. Import Required Libraries

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from nr5g_waveform import (
    NR5GConfig, generate_nr5g_waveform, save_waveform_iq,
    qam_modulate, qam_demodulate_hard, _qam_constellation,
    ofdm_modulate, ofdm_demodulate, generate_dmrs, _cp_lengths,
)
from nr5g_demod import demodulate_nr5g, channel_estimate_dmrs, equalise_zf, compute_evm
from vsa_89600 import VSA89600, simulate_vsa_result, run_vsa_demod

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
print("All imports OK")

## 2. Define 5G NR Parameters

Configure the waveform: numerology μ=1 (30 kHz SCS), 20 MHz BW, 51 RB, 64-QAM, 2 slots.
SNR is set to 30 dB for a clean but realistic signal.

In [ ]:
cfg = NR5GConfig(
    mu=1,                # numerology: SCS = 30 kHz
    bw_mhz=20.0,        # channel bandwidth
    n_rb=51,             # resource blocks
    modulation="64QAM",  # modulation scheme
    n_slots=2,           # number of slots
    n_fft=2048,          # FFT size
    sample_rate_mhz=61.44,
    dmrs_symbol=2,       # DMRS on OFDM symbol 2
    cell_id=0,
    snr_db=30.0,         # AWGN SNR
    seed=42,
)

print(f"Numerology:     μ = {cfg.mu} → SCS = {cfg.scs_khz} kHz")
print(f"Bandwidth:      {cfg.bw_mhz} MHz")
print(f"Resource Blocks: {cfg.n_rb} ({cfg.n_sc} subcarriers)")
print(f"Modulation:     {cfg.modulation} ({cfg.bits_per_symbol} bits/symbol)")
print(f"FFT size:       {cfg.n_fft}")
print(f"Sample rate:    {cfg.sample_rate_mhz} MHz")
print(f"Symbols/slot:   {cfg.symbols_per_slot}")
print(f"Total symbols:  {cfg.symbols_per_slot * cfg.n_slots}")
print(f"SNR:            {cfg.snr_db} dB")

## 3. Generate 5G NR Resource Grid and Symbols

Generate random data bits, apply QAM modulation, insert DMRS on designated OFDM symbols,
and build the full resource grid.

In [ ]:
tx = generate_nr5g_waveform(cfg)

print(f"TX bits:        {len(tx['tx_bits'])}")
print(f"TX symbols:     {len(tx['tx_symbols'])}")
print(f"Resource grid:  {tx['resource_grid'].shape} (symbols × subcarriers)")
print(f"DMRS symbols:   {len(tx['dmrs_symbols'])} OFDM symbols carry DMRS")

# Show resource grid power
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(20 * np.log10(np.abs(tx["resource_grid"]) + 1e-10),
               aspect="auto", cmap="viridis", vmin=-30, vmax=10)
ax.set_xlabel("Subcarrier"); ax.set_ylabel("OFDM Symbol")
ax.set_title("TX Resource Grid (dB)")
plt.colorbar(im, ax=ax, label="Power (dB)")
for sym_idx in tx["dmrs_symbols"]:
    ax.axhline(sym_idx, color="red", linewidth=0.5, linestyle="--", alpha=0.7)
    ax.text(cfg.n_sc + 5, sym_idx, "DMRS", color="red", fontsize=8, va="center")
plt.tight_layout()
plt.show()

## 4. Apply OFDM Modulation — Time-Domain Waveform

IFFT each OFDM symbol, add cyclic prefix, concatenate into baseband IQ waveform.

In [ ]:
time_signal = tx["time_signal"]
time_clean = tx["time_signal_clean"]

print(f"Waveform length:  {len(time_signal)} samples")
print(f"Duration:         {len(time_signal) / (cfg.sample_rate_mhz * 1e6) * 1e6:.1f} µs")
print(f"Peak-to-Average:  {10*np.log10(np.max(np.abs(time_signal)**2) / np.mean(np.abs(time_signal)**2)):.1f} dB")

# Time domain and spectrum
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

t_us = np.arange(min(5000, len(time_signal))) / (cfg.sample_rate_mhz)
ax1.plot(t_us, time_signal[:len(t_us)].real, linewidth=0.4, label="I")
ax1.plot(t_us, time_signal[:len(t_us)].imag, linewidth=0.4, label="Q", alpha=0.7)
ax1.set_xlabel("Time (µs)"); ax1.set_ylabel("Amplitude")
ax1.set_title("Time-Domain Waveform (first 5000 samples)")
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# Spectrum
N = len(time_signal)
freq = np.fft.fftshift(np.fft.fftfreq(N, 1 / (cfg.sample_rate_mhz)))
spectrum = 20 * np.log10(np.abs(np.fft.fftshift(np.fft.fft(time_signal))) / N + 1e-15)
ax2.plot(freq, spectrum, linewidth=0.3)
ax2.set_xlabel("Frequency (MHz)"); ax2.set_ylabel("Power (dB)")
ax2.set_title("Waveform Spectrum"); ax2.grid(True, alpha=0.3)
ax2.set_xlim([-cfg.bw_mhz, cfg.bw_mhz])

plt.tight_layout(); plt.show()

# Save for VSA
os.makedirs("results", exist_ok=True)
iq_file = "results/nr5g_waveform.bin"
save_waveform_iq(iq_file, time_signal)
save_waveform_iq("results/nr5g_waveform.csv", time_signal, fmt="csv")
print(f"\nSaved IQ waveform: {iq_file} ({os.path.getsize(iq_file)} bytes)")

## 5. Implement Custom 5G NR Demodulation Code

The demod pipeline:
1. **OFDM demodulation** — remove CP, FFT each symbol, extract subcarriers
2. **Channel estimation** — LS estimate from DMRS, interpolate across subcarriers
3. **ZF equalization** — divide received grid by channel estimate
4. **Symbol demapping** — nearest-neighbour hard decision to recover QAM symbols and bits

In [ ]:
# Step-by-step demod to show intermediate results
cp = _cp_lengths(cfg.n_fft, cfg.mu) * cfg.n_slots
n_sym_total = cfg.symbols_per_slot * cfg.n_slots

# 5a. OFDM demodulate
rx_grid = ofdm_demodulate(time_signal, cfg.n_fft, cfg.n_sc, cp, n_sym_total)
print(f"Received grid shape: {rx_grid.shape}")

# 5b. Channel estimation from DMRS
h_est = channel_estimate_dmrs(rx_grid, cfg)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.imshow(20 * np.log10(np.abs(rx_grid) + 1e-10), aspect="auto", cmap="viridis", vmin=-30, vmax=10)
ax1.set_title("Received Resource Grid (dB)")
ax1.set_xlabel("Subcarrier"); ax1.set_ylabel("OFDM Symbol")

# Show channel estimate magnitude & phase for slot 0 DMRS symbol
ax2.plot(np.abs(h_est[cfg.dmrs_symbol, :]), label="|H|", linewidth=0.8)
ax2.plot(np.angle(h_est[cfg.dmrs_symbol, :]), label="∠H", linewidth=0.8, alpha=0.7)
ax2.set_title("Channel Estimate (DMRS symbol)")
ax2.set_xlabel("Subcarrier"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 5c. ZF Equalization
eq_grid = equalise_zf(rx_grid, h_est)
print(f"Equalised grid shape: {eq_grid.shape}")

## 6. Demodulate the Waveform Using Custom Code

Run the full demod pipeline and compute EVM / BER.

In [ ]:
py_result = demodulate_nr5g(
    time_signal, cfg,
    tx_bits=tx["tx_bits"],
    tx_symbols=tx["tx_symbols"],
    data_positions=tx["data_positions"],
)

print("=== Python Demodulation Results ===")
print(f"  EVM RMS:       {py_result.evm_rms:.4f} %")
print(f"  EVM Peak:      {np.max(py_result.evm_per_symbol):.4f} %")
print(f"  BER:           {py_result.ber:.2e}")
print(f"  Data symbols:  {len(py_result.rx_symbols)}")
print(f"  Bits decoded:  {len(py_result.rx_bits)}")

## 7. Configure and Connect to Keysight VSA 89600

Set `USE_REAL_VSA = True` if VSA 89600 software is installed on this machine.
Otherwise we simulate VSA results using our demod engine with small perturbations.

In [ ]:
USE_REAL_VSA = False  # Set True if Keysight VSA 89600 is installed

if USE_REAL_VSA:
    try:
        import win32com.client
        vsa = VSA89600(visible=True)
        vsa.connect()
        print("Connected to VSA 89600")
    except Exception as e:
        print(f"Cannot connect to VSA 89600: {e}")
        print("Falling back to simulated mode")
        USE_REAL_VSA = False
else:
    print("Running in OFFLINE mode (simulated VSA results)")

## 8. Load Waveform into VSA 89600 and Demodulate

Load the IQ binary file, configure 5G NR demod parameters, and run the analysis.

In [ ]:
if USE_REAL_VSA:
    vsa.load_iq_binary(
        os.path.abspath(iq_file),
        sample_rate_hz=cfg.sample_rate_mhz * 1e6,
        center_freq_hz=3.5e9,
    )
    vsa.setup_nr5g_demod(
        mu=cfg.mu, bw_mhz=cfg.bw_mhz, n_rb=cfg.n_rb,
        modulation=cfg.modulation, cell_id=cfg.cell_id,
    )
    import time; time.sleep(3)
    print("VSA 89600: waveform loaded and 5G NR demod configured")
else:
    print("Simulating VSA 89600 demodulation...")

## 9. Extract Demodulation Results from VSA 89600

Retrieve constellation data, EVM, frequency error from the VSA.

In [ ]:
if USE_REAL_VSA:
    vsa_result = vsa.get_results()
    vsa.disconnect()
else:
    vsa_result = simulate_vsa_result(time_signal, cfg)

print("=== VSA 89600 Results ===")
print(f"  EVM RMS:       {vsa_result.evm_rms:.4f} %")
print(f"  EVM Peak:      {vsa_result.evm_peak:.4f} %")
print(f"  Freq Error:    {vsa_result.freq_error_hz:.2f} Hz")
print(f"  Constellation: {len(vsa_result.symbols_i)} symbols")

## 10. Compare Custom Demodulation vs VSA 89600 Results

Quantitative comparison: EVM difference, symbol correlation, metric summary table.

In [ ]:
# Symbol-level correlation
py_sym = py_result.rx_symbols
vsa_sym = vsa_result.symbols_i + 1j * vsa_result.symbols_q
n = min(len(py_sym), len(vsa_sym))

corr = np.abs(np.corrcoef(py_sym[:n].real, vsa_sym[:n].real)[0, 1])
sym_diff_rms = np.sqrt(np.mean(np.abs(py_sym[:n] - vsa_sym[:n]) ** 2))

print("=" * 60)
print("  Quantitative Comparison")
print("=" * 60)
print(f"  {'Metric':<35s} {'Python':>10s} {'VSA':>10s}")
print("-" * 60)
print(f"  {'EVM RMS (%)':<35s} {py_result.evm_rms:>10.4f} {vsa_result.evm_rms:>10.4f}")
print(f"  {'EVM Peak (%)':<35s} {np.max(py_result.evm_per_symbol):>10.4f} {vsa_result.evm_peak:>10.4f}")
print(f"  {'Freq Error (Hz)':<35s} {'N/A':>10s} {vsa_result.freq_error_hz:>10.2f}")
if py_result.ber is not None:
    print(f"  {'BER':<35s} {py_result.ber:>10.2e} {'N/A':>10s}")
print(f"  {'# Symbols':<35s} {len(py_sym):>10d} {len(vsa_sym):>10d}")
print("-" * 60)
print(f"  {'EVM Δ (Py - VSA)':<35s} {py_result.evm_rms - vsa_result.evm_rms:>+10.4f} %")
print(f"  {'Symbol correlation (I)':<35s} {corr:>10.6f}")
print(f"  {'Symbol diff RMS':<35s} {sym_diff_rms:>10.6f}")
print("=" * 60)

## 11. Visualize Constellation Diagrams Side by Side

In [ ]:
ref = _qam_constellation(cfg.bits_per_symbol)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Python constellation
ax = axes[0]
ax.scatter(py_sym.real, py_sym.imag, s=0.5, alpha=0.25, c="steelblue")
ax.scatter(ref.real, ref.imag, s=50, c="red", marker="+", linewidths=1.5)
ax.set_title(f"Python Demod\nEVM = {py_result.evm_rms:.3f}%", fontsize=11)
ax.set_xlabel("I"); ax.set_ylabel("Q")
ax.set_aspect("equal"); ax.grid(True, alpha=0.3)

# VSA constellation
ax = axes[1]
ax.scatter(vsa_result.symbols_i, vsa_result.symbols_q, s=0.5, alpha=0.25, c="darkorange")
ax.scatter(ref.real, ref.imag, s=50, c="red", marker="+", linewidths=1.5)
ax.set_title(f"VSA 89600\nEVM = {vsa_result.evm_rms:.3f}%", fontsize=11)
ax.set_xlabel("I"); ax.set_ylabel("Q")
ax.set_aspect("equal"); ax.grid(True, alpha=0.3)

# Overlay
ax = axes[2]
ax.scatter(py_sym[:n].real, py_sym[:n].imag, s=0.5, alpha=0.2, c="steelblue", label="Python")
ax.scatter(vsa_sym[:n].real, vsa_sym[:n].imag, s=0.5, alpha=0.2, c="darkorange", label="VSA")
ax.scatter(ref.real, ref.imag, s=50, c="red", marker="+", linewidths=1.5, label="Ideal")
ax.set_title("Overlay Comparison", fontsize=11)
ax.set_xlabel("I"); ax.set_ylabel("Q")
ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, markerscale=5)

plt.suptitle(f"5G NR {cfg.modulation} Constellation — μ={cfg.mu}, {cfg.bw_mhz}MHz, SNR={cfg.snr_db}dB",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("results/constellation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()